# Topic 2 – API Design Principles in Practice 🧪
### Test cases that prove good (and bad) design, using game APIs

In Topic 1 we *consumed* APIs. Now we **judge their design**. Each step turns one principle from the
*API Design Principles* slides into a **test case** with a visible `PASS` / `FAIL`.

| Slide principle | Test case in this notebook |
|---|---|
| API is a contract | Same fields in every response (schema test) |
| Anatomy of a URL | Parse a real URL into scheme, host, path, query |
| Nouns, not verbs | Naming test on real resource names |
| Methods = CRUD | GET / POST / PUT / PATCH / DELETE against httpbin |
| Consistent schemas | Compare several monsters field by field |
| Right status code | 404 test + a decision function (same as the flowchart) |
| Pagination | Walk pages and check there is no overlap |
| Versioning | `/v2`, `/api/2014` style paths |
| Final checklist | An automatic **design scorecard** |

Services (free, no key): [PokéAPI](https://pokeapi.co), [D&D 5e API](https://www.dnd5eapi.co), [httpbin.org](https://httpbin.org).

## Step -1 – Install dependencies (run once)
This installs `requests` into **the kernel you selected**, so it works in any environment (venv, system, Cursor, VS Code, Jupyter).

In [ ]:
%pip install -q requests

## Step 0 – Setup and a tiny test helper
`check()` prints PASS/FAIL and keeps a score, so every step reads like a real test report.

In [1]:
import requests
import json
from urllib.parse import urlparse, parse_qs

RESULTS = []          # (name, passed)


def check(name, condition, detail=""):
    """Record and print one test case."""
    RESULTS.append((name, bool(condition)))
    icon = "PASS" if condition else "FAIL"
    print(f"[{icon}] {name}" + (f"  -> {detail}" if detail else ""))


def get_json(url, **kwargs):
    """Safe GET from Topic 1: timeout + clear errors. Returns (status, json_or_None)."""
    try:
        r = requests.get(url, timeout=10, **kwargs)
        try:
            return r.status_code, r.json()
        except ValueError:
            return r.status_code, None
    except requests.exceptions.RequestException as e:
        print("Network problem:", e)
        return None, None


POKE = "https://pokeapi.co/api/v2"
DND = "https://www.dnd5eapi.co/api"
print("Ready")

Ready


## Step 1 – An API is a contract (schema test)
**Principle:** a developer should be able to *guess* the shape of the next response.
**Test:** three different Pokémon must expose the same top-level fields.

In [2]:
names = ["pikachu", "charmander", "squirtle"]
required = {"id", "name", "height", "weight", "stats", "types"}

for n in names:
    status, body = get_json(f"{POKE}/pokemon/{n}")
    check(f"'{n}' answers 200", status == 200, f"status={status}")
    check(f"'{n}' has the contract fields", required.issubset(body.keys()),
          f"missing={required - set(body.keys()) or 'none'}")

[PASS] 'pikachu' answers 200  -> status=200
[PASS] 'pikachu' has the contract fields  -> missing=none


[PASS] 'charmander' answers 200  -> status=200
[PASS] 'charmander' has the contract fields  -> missing=none


[PASS] 'squirtle' answers 200  -> status=200
[PASS] 'squirtle' has the contract fields  -> missing=none


## Step 2 – Anatomy of a URL
**Principle:** the *path* identifies the resource; the *query* only filters or tweaks it.
`urllib.parse` splits any URL so we can test each part.

In [3]:
url = "https://pokeapi.co/api/v2/pokemon?limit=5&offset=10"
parts = urlparse(url)

print("scheme :", parts.scheme)
print("host   :", parts.netloc)
print("path   :", parts.path)
print("query  :", parse_qs(parts.query))

check("Uses HTTPS", parts.scheme == "https")
check("Version is in the path", "/v2/" in parts.path)
check("Query holds only filters", set(parse_qs(parts.query)) <= {"limit", "offset"})

scheme : https
host   : pokeapi.co
path   : /api/v2/pokemon
query  : {'limit': ['5'], 'offset': ['10']}
[PASS] Uses HTTPS
[PASS] Version is in the path
[PASS] Query holds only filters


## Step 3 – Nouns, not verbs (naming test)
**Principle:** `GET /models` good, `GET /getAllModels` bad.
**Test:** read the *real* resource names of two APIs and look for verbs, capitals or underscores.

In [4]:
VERB_PREFIXES = ("get", "create", "delete", "update", "fetch", "set", "remove")


def is_good_resource_name(name):
    """lowercase-with-hyphens, no verb prefix."""
    return (name == name.lower()
            and "_" not in name
            and not name.startswith(VERB_PREFIXES))


# our own examples first (the slide)
for bad in ["getAllModels", "createPrediction", "delete_model"]:
    check(f"'{bad}' is rejected as a bad name", not is_good_resource_name(bad))
for good in ["models", "predictions", "model-versions"]:
    check(f"'{good}' is accepted as a good name", is_good_resource_name(good))

[PASS] 'getAllModels' is rejected as a bad name
[PASS] 'createPrediction' is rejected as a bad name
[PASS] 'delete_model' is rejected as a bad name
[PASS] 'models' is accepted as a good name
[PASS] 'predictions' is accepted as a good name
[PASS] 'model-versions' is accepted as a good name


In [5]:
# now the real services
for label, root in [("PokéAPI", POKE), ("D&D 5e", DND)]:
    status, resources = get_json(root)
    bad = [r for r in resources if not is_good_resource_name(r)]
    check(f"{label}: all {len(resources)} resource names follow the rule", not bad,
          f"offenders={bad or 'none'}")

[PASS] PokéAPI: all 51 resource names follow the rule  -> offenders=none


[PASS] D&D 5e: all 23 resource names follow the rule  -> offenders=none


## Step 3b – Resource tree
The URL structure is a tree. Let's print the first resources of the D&D API like a mini documentation.

In [6]:
status, root = get_json(DND)
print("/api")
for name, path in list(root.items())[:6]:
    print(f"  |-- {name:<16} {path}")

/api


  |-- ability-scores   /api/2014/ability-scores
  |-- alignments       /api/2014/alignments
  |-- backgrounds      /api/2014/backgrounds
  |-- classes          /api/2014/classes
  |-- conditions       /api/2014/conditions
  |-- damage-types     /api/2014/damage-types


## Step 4 – HTTP methods as CRUD
**Principle:** the method is the verb. **Test:** httpbin accepts the right method on each endpoint and rejects the wrong one (`405 Method Not Allowed`).

In [7]:
BIN = "https://httpbin.org"
player = {"player": "Luis", "level": 7}

cases = [
    ("GET",    "/get",    None),
    ("POST",   "/post",   player),
    ("PUT",    "/put",    player),
    ("PATCH",  "/patch",  {"level": 8}),
    ("DELETE", "/delete", None),
]
for method, path, payload in cases:
    r = requests.request(method, BIN + path, json=payload, timeout=10)
    check(f"{method:<6} {path:<8} -> 200", r.status_code == 200, f"status={r.status_code}")

# wrong method on purpose
r = requests.get(BIN + "/post", timeout=10)
check("GET on a POST-only endpoint -> 405", r.status_code == 405, f"status={r.status_code}")

[PASS] GET    /get     -> 200  -> status=200


[PASS] POST   /post    -> 200  -> status=200


[PASS] PUT    /put     -> 200  -> status=200


[PASS] PATCH  /patch   -> 200  -> status=200


[PASS] DELETE /delete  -> 200  -> status=200


[PASS] GET on a POST-only endpoint -> 405  -> status=405


## Step 5 – Consistent schemas
**Principle:** same field names and types everywhere.
**Test:** compare several monsters; every one must have the same keys with the same types.

In [8]:
monsters = ["goblin", "orc", "wolf", "zombie"]
fields = {"index": str, "name": str, "hit_points": int, "armor_class": list, "actions": list}

for m in monsters:
    status, body = get_json(f"{DND}/monsters/{m}")
    ok = status == 200 and all(isinstance(body.get(k), t) for k, t in fields.items())
    check(f"'{m}' matches the schema", ok)

[PASS] 'goblin' matches the schema


[PASS] 'orc' matches the schema


[PASS] 'wolf' matches the schema


[PASS] 'zombie' matches the schema


In [9]:
# Naming consistency inside the response: snake_case keys only
status, body = get_json(f"{DND}/monsters/goblin")
bad_keys = [k for k in body if k != k.lower() or " " in k]
check("All keys are snake_case", not bad_keys, f"offenders={bad_keys or 'none'}")

[PASS] All keys are snake_case  -> offenders=none


## Step 6 – The right status code + uniform errors
**Principle:** a missing resource is `404`, bad input is `4xx`, a server crash is `5xx`.
**Test 1:** real APIs. **Test 2:** our decision function must match the flowchart from the slides.

In [10]:
status, _ = get_json(f"{POKE}/pokemon/not-a-real-pokemon")
check("PokéAPI: unknown resource -> 404", status == 404, f"status={status}")

status, _ = get_json(f"{DND}/monsters/not-a-real-monster")
check("D&D API: unknown resource -> 404", status == 404, f"status={status}")

for code_ in [400, 401, 404, 500, 503]:
    r = requests.get(f"{BIN}/status/{code_}", timeout=10)
    check(f"httpbin simulates {code_}", r.status_code == code_)

[PASS] PokéAPI: unknown resource -> 404  -> status=404


[PASS] D&D API: unknown resource -> 404  -> status=404


[PASS] httpbin simulates 400


[PASS] httpbin simulates 401


[PASS] httpbin simulates 404


[PASS] httpbin simulates 500


[PASS] httpbin simulates 503


In [11]:
def choose_status(valid, exists, server_ok, created=False):
    """Same logic as the flowchart: valid? -> exists? -> server worked?"""
    if not valid:
        return 422
    if not exists:
        return 404
    if not server_ok:
        return 500
    return 201 if created else 200


check("invalid input        -> 422", choose_status(False, True, True) == 422)
check("missing resource     -> 404", choose_status(True, False, True) == 404)
check("server failure       -> 500", choose_status(True, True, False) == 500)
check("happy path           -> 200", choose_status(True, True, True) == 200)
check("happy path + created -> 201", choose_status(True, True, True, created=True) == 201)

[PASS] invalid input        -> 422
[PASS] missing resource     -> 404
[PASS] server failure       -> 500
[PASS] happy path           -> 200
[PASS] happy path + created -> 201


In [12]:
def error_body(code, message):
    """One standard error shape for the whole API."""
    return {"code": code, "message": message}


err = error_body("MODEL_NOT_FOUND", "Model 99 does not exist")
check("Error has a machine-readable code", "code" in err and err["code"].isupper())
check("Error has a human message", isinstance(err["message"], str) and len(err["message"]) > 5)
print(json.dumps(err, indent=2))

[PASS] Error has a machine-readable code
[PASS] Error has a human message
{
  "code": "MODEL_NOT_FOUND",
  "message": "Model 99 does not exist"
}


## Step 7 – Pagination
**Principle:** never return everything; return a page plus navigation info.
**Test:** walk three pages of the PokéAPI and make sure no Pokémon repeats.

In [13]:
limit = 5
seen = []
url = f"{POKE}/pokemon?limit={limit}&offset=0"

for page in range(1, 4):
    status, body = get_json(url)
    names_on_page = [p["name"] for p in body["results"]]
    check(f"Page {page}: returns exactly {limit} items", len(names_on_page) == limit)
    check(f"Page {page}: no overlap with previous pages", not set(names_on_page) & set(seen),
          names_on_page)
    seen += names_on_page
    url = body["next"]            # the API itself tells us where the next page is

check("Envelope contains count/next/previous/results",
      {"count", "next", "previous", "results"} <= set(body.keys()))

[PASS] Page 1: returns exactly 5 items
[PASS] Page 1: no overlap with previous pages  -> ['bulbasaur', 'ivysaur', 'venusaur', 'charmander', 'charmeleon']


[PASS] Page 2: returns exactly 5 items
[PASS] Page 2: no overlap with previous pages  -> ['charizard', 'squirtle', 'wartortle', 'blastoise', 'caterpie']


[PASS] Page 3: returns exactly 5 items
[PASS] Page 3: no overlap with previous pages  -> ['metapod', 'butterfree', 'weedle', 'kakuna', 'beedrill']
[PASS] Envelope contains count/next/previous/results


In [14]:
# Our own pagination (same as the slide) tested with edge cases
HISTORY = [{"id": i, "price": 300000 + i * 1000} for i in range(1, 16)]


def paginate(items, page=1, limit=5):
    start = (page - 1) * limit
    return {"items": items[start:start + limit], "page": page,
            "limit": limit, "total": len(items)}


check("page 1 has ids 1-5", [x["id"] for x in paginate(HISTORY, 1)["items"]] == [1, 2, 3, 4, 5])
check("page 3 has ids 11-15", [x["id"] for x in paginate(HISTORY, 3)["items"]] == [11, 12, 13, 14, 15])
check("page beyond the end is empty", paginate(HISTORY, 4)["items"] == [])
check("total is always the full size", paginate(HISTORY, 2)["total"] == 15)

[PASS] page 1 has ids 1-5
[PASS] page 3 has ids 11-15
[PASS] page beyond the end is empty
[PASS] total is always the full size


## Step 8 – Versioning
**Principle:** change without breaking clients: `/v1`, `/v2`.
**Test:** the version lives in the path, and the old version keeps working.

In [15]:
check("PokéAPI version in path (/v2)", "/v2" in POKE)

# D&D API serves explicit editions
for edition in ["2014", "2024"]:
    status, body = get_json(f"{DND}/{edition}")
    check(f"D&D edition '{edition}' root is reachable", status == 200, f"status={status}")

status, goblin = get_json(f"{DND}/2014/monsters/goblin")
check("Old edition still answers (backward compatible)", status == 200 and goblin["name"] == "Goblin")

[PASS] PokéAPI version in path (/v2)


[PASS] D&D edition '2014' root is reachable  -> status=200


[PASS] D&D edition '2024' root is reachable  -> status=200


[PASS] Old edition still answers (backward compatible)


## Step 9 – Design scorecard
We bundle the principles into one function. Give it any API and it returns a score, so you can **audit APIs automatically**.

In [16]:
def scorecard(label, root_url, sample_ok_url, sample_missing_url, list_url):
    results = []

    status, resources = get_json(root_url)
    results.append(("Resource names follow naming rule",
                    status == 200 and all(is_good_resource_name(r) for r in resources)))

    status, _ = get_json(sample_ok_url)
    results.append(("Valid resource -> 200", status == 200))

    status, _ = get_json(sample_missing_url)
    results.append(("Missing resource -> 404", status == 404))

    status, body = get_json(list_url)
    results.append(("List is paginated (has count/results)",
                    isinstance(body, dict) and "count" in body and "results" in body))

    results.append(("Uses HTTPS", root_url.startswith("https://")))

    passed = sum(ok for _, ok in results)
    print(f"\n=== {label}: {passed}/{len(results)} ===")
    for name, ok in results:
        print(f"  [{'PASS' if ok else 'FAIL'}] {name}")
    return passed, len(results)


scorecard("PokéAPI", POKE, f"{POKE}/pokemon/pikachu", f"{POKE}/pokemon/zzz", f"{POKE}/pokemon?limit=2")
scorecard("D&D 5e API", DND, f"{DND}/monsters/goblin", f"{DND}/monsters/zzz", f"{DND}/monsters?name=dragon")


=== PokéAPI: 5/5 ===
  [PASS] Resource names follow naming rule
  [PASS] Valid resource -> 200
  [PASS] Missing resource -> 404
  [PASS] List is paginated (has count/results)
  [PASS] Uses HTTPS



=== D&D 5e API: 5/5 ===
  [PASS] Resource names follow naming rule
  [PASS] Valid resource -> 200
  [PASS] Missing resource -> 404
  [PASS] List is paginated (has count/results)
  [PASS] Uses HTTPS


(5, 5)

## Step 10 – Design our own game API (paper contract + tests)
Before writing any FastAPI code (Topic 3), we write the **contract** and test it with plain Python.
Idea: an API that predicts a game character's **power** from its stats (our future ML endpoint).

In [17]:
CONTRACT = {
    "GET  /v1/characters":        "list characters (paginated)",
    "GET  /v1/characters/{id}":   "one character",
    "POST /v1/predictions":       "predict power from stats",
}

REQUEST_FIELDS = {"attack": int, "defense": int, "speed": int}


def validate_request(payload):
    """Returns (status, body) following our design rules."""
    for field, typ in REQUEST_FIELDS.items():
        if field not in payload or not isinstance(payload[field], typ):
            return 422, error_body("INVALID_INPUT", f"'{field}' must be a {typ.__name__}")
    return 200, {"predicted_power": 1.5 * payload["attack"] + payload["defense"] + 0.5 * payload["speed"],
                 "model_version": "0.1.0"}


for route in CONTRACT:
    method, path = route.split()
    check(f"{route}: path uses versioned plural noun", path.startswith("/v1/") and is_good_resource_name(path.split("/")[2]))

check("valid request -> 200", validate_request({"attack": 55, "defense": 40, "speed": 90})[0] == 200)
check("missing field -> 422", validate_request({"attack": 55})[0] == 422)
check("text instead of number -> 422", validate_request({"attack": "high", "defense": 1, "speed": 1})[0] == 422)
check("422 uses the standard error shape", set(validate_request({})[1]) == {"code", "message"})
check("success includes model_version", "model_version" in validate_request({"attack": 1, "defense": 1, "speed": 1})[1])

[PASS] GET  /v1/characters: path uses versioned plural noun
[PASS] GET  /v1/characters/{id}: path uses versioned plural noun
[PASS] POST /v1/predictions: path uses versioned plural noun
[PASS] valid request -> 200
[PASS] missing field -> 422
[PASS] text instead of number -> 422
[PASS] 422 uses the standard error shape
[PASS] success includes model_version


## Final report

In [18]:
passed = sum(ok for _, ok in RESULTS)
total = len(RESULTS)
print(f"{passed}/{total} test cases passed ({passed / total:.0%})")
failed = [n for n, ok in RESULTS if not ok]
print("Failed:", failed or "none")

65/65 test cases passed (100%)
Failed: none


### Exercises
1. Add a test that every PokéAPI list response has `next` and `previous` as either `None` or a URL.
2. Write `is_good_field_name()` and test it against the keys of a Pokémon response (what do you find?).
3. Extend `scorecard()` with a *versioning* check and a *response time under 2 seconds* check.
4. Add `PATCH /v1/characters/{id}` to `CONTRACT` and write 2 tests for it.

**Next:** Topic 3 – FastAPI. We turn this paper contract into a real running server.